In [ ]:
# Prueba Scraping PcComponentes con Playwright
# Usamos Playwright porque la web carga con JavaScript y tiene protección antibot.

SyntaxError: invalid syntax (2544792687.py, line 2)

In [13]:
from playwright.async_api import async_playwright
import pandas as pd
from datetime import datetime

In [14]:
async def scrape_pccomponentes():
    productos = []
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        await page.goto(
            "https://www.pccomponentes.com/portatiles")
        
        # Espera a que carguen los productos
        await page.wait_for_selector(
            "h3.product-card__title", 
            timeout=15000)
        
        nombres = await page.query_selector_all(
                    "h3.product-card__title")
        precios = await page.query_selector_all(
                    "[data-e2e='price-card']")
        precios_orig = await page.query_selector_all(
                    "[data-e2e='crossedPrice']")
        
        print(f"Nombres encontrados: {len(nombres)}")
        print(f"Precios encontrados: {len(precios)}")
        
        for i, nombre in enumerate(nombres):
            precio_actual = (
                await precios[i].inner_text() 
                if i < len(precios) else None)
            precio_original = (
                await precios_orig[i].inner_text() 
                if i < len(precios_orig) else None)
            
            productos.append({
                "nombre": await nombre.inner_text(),
                "precio_actual": precio_actual,
                "precio_original": precio_original,
                "plataforma": "PcComponentes",
                "fecha": datetime.now().strftime(
                            "%Y-%m-%d %H:%M")
            })
        
        await browser.close()
    
    return pd.DataFrame(productos)

In [24]:
from playwright_stealth import Stealth

async def diagnostico_stealth():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,  # Visible para ver qué pasa
            args=["--disable-blink-features=AutomationControlled"]
        )
        
        context = await browser.new_context(
            viewport={"width": 1280, "height": 800},
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            locale="es-ES",
        )
        
        # API nueva de playwright-stealth
        stealth = Stealth()
        await stealth.apply_stealth_async(context)
        
        page = await context.new_page()
        
        await page.goto(
            "https://www.pccomponentes.com/portatiles",
            wait_until="domcontentloaded",
            timeout=30000)
        
        # Esperamos más tiempo simulando comportamiento humano
        await page.wait_for_timeout(8000)
        
        html = await page.content()
        
        print(f"Tamaño del HTML: {len(html)} caracteres")
        print(f"¿Contiene 'product-card': {'product-card' in html}")
        print(f"¿Contiene 'Portátil': {'Portátil' in html}")
        print(f"¿Contiene 'robot': {'robot' in html.lower()}")
        
        with open("../data/raw/pagina_pccomponentes.html", "w", encoding="utf-8") as f:
            f.write(html)
        print("HTML guardado en data/raw/pagina_pccomponentes.html")
        
        await browser.close()

await diagnostico_stealth()

Tamaño del HTML: 1196018 caracteres
¿Contiene 'product-card': True
¿Contiene 'Portátil': True
¿Contiene 'robot': True
HTML guardado en data/raw/pagina_pccomponentes.html


In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
from pathlib import Path

def scrape_idealo():
    url = "https://www.idealo.es/cat/3394/portatiles.html"
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "es-ES,es;q=0.9",
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
    except requests.RequestException as e:
        print(f"Error de conexión con Idealo: {e}")
        return pd.DataFrame()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    raw_dir = Path("../data/raw")
    raw_dir.mkdir(parents=True, exist_ok=True)
    html_path = raw_dir / f"idealo_portatiles_{timestamp}.html"
    html_path.write_text(response.text, encoding="utf-8")

    print(f"Status: {response.status_code}")
    print(f"Tamaño HTML: {len(response.text)}")
    print(f"HTML guardado en: {html_path}")

    texto_lower = response.text.lower()
    bloqueado = any(k in texto_lower for k in ["akamai", "sorry", "reference id", "captcha", "forbidden", "access denied"])
    print(f"¿Posible bloqueo anti-bot?: {bloqueado}")

    if response.status_code != 200 or bloqueado:
        print("Plan D bloqueado en este entorno/red. Se recomienda estrategia alternativa.")
        return pd.DataFrame()

    soup = BeautifulSoup(response.text, "html.parser")
    items = soup.select("article, .offerList-item, .sr-resultList__item")
    productos = []

    for item in items:
        nombre = item.get_text(" ", strip=True)[:160]
        precio_tag = item.select_one("[class*='price'], [data-testid*='price']")
        precio = precio_tag.get_text(" ", strip=True) if precio_tag else None

        if nombre and precio:
            productos.append({
                "nombre": nombre,
                "precio_actual": precio,
                "plataforma": "Idealo",
                "fecha": datetime.now().strftime("%Y-%m-%d %H:%M")
            })

    df = pd.DataFrame(productos)
    print(f"Productos extraídos: {len(df)}")
    return df

df_idealo = scrape_idealo()
df_idealo.head()

Error de conexión con Idealo: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


""


In [25]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from scraper import run_daily_scrape

# Ejecución diaria (ajusta headless=False si quieres ver el navegador)
df_diario = await run_daily_scrape(max_items_per_platform=20, headless=True)
df_diario.head()

Archivo guardado: /Users/david/Documents/scraper/data/raw/precios_portatiles_20260330_1251.csv
plataforma
PcComponentes    20
Amazon           20
Name: count, dtype: int64


,nombre,precio_actual,precio_original,descuento,valoracion,plataforma,fecha
0,Portátil Alurin Flex Advance AMD Ryzen 7-5825U...,"489,99€","644,99€",None,NaN,PcComponentes,2026-03-30 12:51
1,"PcCom Revolt 5070 Intel Core i7-14650HX 16""/QH...","1.658,99€","2.079,99€",None,NaN,PcComponentes,2026-03-30 12:51
2,"Portátil Lenovo LOQ 15IRX10 15.6"" Intel Core i...",1.149€,1.499€,None,NaN,PcComponentes,2026-03-30 12:51
3,"Portátil Lenovo Legion 5 15AHP10 15.1"" AMD Ryz...",1.289€,1.699€,None,NaN,PcComponentes,2026-03-30 12:51
4,Apple Macbook Air Apple M4 10 Núcleos/16 GB/25...,899€,1.179€,None,NaN,PcComponentes,2026-03-30 12:51


In [17]:
async def prueba_elcorteingles_final(config: ScrapeConfig):
    context = await _new_context(config)
    page = await context.new_page()
    rows = []
    
    try:
        print("Cargando El Corte Inglés...")
        await page.goto("https://www.elcorteingles.es/informatica/portatiles/", wait_until="domcontentloaded", timeout=45000)
        
        print("Espera pasiva para carga de scripts...")
        await page.wait_for_timeout(4000)

        # --- 1. LIMPIEZA DE INTERFAZ ---
        try:
            await page.locator("#onetrust-accept-btn-handler").click(timeout=2000, force=True)
            print("✅ Cookies aceptadas.")
        except: pass

        try:
            await page.get_by_text("Nuevo IA Comparador", exact=False).first.click(timeout=2000, force=True)
            await page.keyboard.press("Escape")
            print("✅ Tooltip IA cerrado.")
        except: pass

        try:
            await page.evaluate("""
                document.querySelectorAll('iframe, div[class*="chat"], div[class*="bot"]').forEach(e => e.remove());
            """)
            print("✅ Chatbot neutralizado vía JS.")
        except: pass

        # --- 2. SCROLL PARA CARGAR PRECIOS (Lazy Loading) ---
        print("Haciendo scroll para cargar imágenes y precios...")
        for _ in range(4):
            await page.mouse.wheel(0, 800)
            await page.wait_for_timeout(1000)

        # --- 3. EXTRACCIÓN QUIRÚRGICA (Basada en tu auditoría HTML) ---
        cards = page.locator("article.product_preview")
        total = await cards.count()
        # Limitamos a lo que ponga la configuración
        limit = min(total, config.max_items_per_platform)
        print(f"Tarjetas detectadas ECI: {limit} de {total} posibles")

        for i in range(limit):
            card = cards.nth(i)
            
            # Nombre: Usamos el selector exacto de tu HTML
            name = await _safe_text(card.locator("a.product_preview-title"))
            
            # Precio Actual: price-sale (rebajado) o price-unit--normal (normal)
            current_price = await _safe_text(card.locator(".price-sale, .price-unit--normal"))
            
            # Precio Original (Bonus que encontramos en tu HTML)
            original_price = await _safe_text(card.locator(".price-unit--original"))
            
            if name and current_price:
                rows.append({
                    "nombre": name, 
                    "precio_actual": current_price, 
                    "precio_original": original_price,
                    "plataforma": "ElCorteIngles"
                })
                
    except Exception as e:
        print(f"❌ Error crítico en ECI: {e}")
    finally:
        await _close_context(context)
        
    return rows

# Ejecutar la prueba
config_test = ScrapeConfig(headless=False)
datos_eci = await prueba_elcorteingles_final(config_test)
df_eci = pd.DataFrame(datos_eci)
display(df_eci)

Cargando El Corte Inglés...
Espera pasiva para carga de scripts...
✅ Tooltip IA cerrado.
✅ Chatbot neutralizado vía JS.
Haciendo scroll para cargar imágenes y precios...
Tarjetas detectadas ECI: 12 de 12 posibles


,nombre,precio_actual,precio_original,plataforma
0,"Portátil Samsung Galaxy Book4 NP750XGJ-KG4ES, ...",Precio de venta 779 €,Precio original 899 €,ElCorteIngles
1,"Portátil Lenovo Yoga Slim 7 14IMH9, Intel Core...",Precio de venta 999 €,Precio original 1.199 €,ElCorteIngles
2,"Tablet Lenovo Idea Tab 8GB + 128GB, 11"", Wi-Fi...",Precio de venta 199 €,NaN,ElCorteIngles
3,"Portátil Gaming HP VICTUS 15-fa2007ns, Intel C...",Precio de venta 1.199 €,Precio original 1.399 €,ElCorteIngles
4,"Monitor PC 68,5cm (27"") LG MyView Smart Monito...",Precio de venta 179 €,Precio original 219 €,ElCorteIngles
5,"Impresora multifunción HP OfficeJet Pro 8134e,...","Precio de venta 139,90 €","Precio original 169,90 €",ElCorteIngles
6,"Portátil HP OmniBook 7 14-fr0001ns, Intel Core...",Precio de venta 899 €,Precio original 999 €,ElCorteIngles
7,"Apple iPad 11"" (2025) A16, Wi-Fi",Precio de venta 359 €,Precio original 379 €,ElCorteIngles
8,"Monitor PC Gaming Samsung Odyssey OLED G6 68,5...",Precio de venta 579 €,Precio original 799 €,ElCorteIngles
9,Impresora Multifunción Epson Expression Home X...,"Precio de venta 60,90 €","Precio original 71,90 €",ElCorteIngles


In [23]:
import sys
import importlib
from pathlib import Path
from datetime import datetime
import pandas as pd

# 1. Rutas
project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# 2. IMPORTANTE: Forzamos a Jupyter a leer el archivo etl.py actualizado
import etl
importlib.reload(etl) # <--- ¡ESTA ES LA MAGIA QUE ARREGLA EL PROBLEMA!
from etl import transform_prices

# 3. Preparar datos
if 'descuento' not in df_eci.columns:
    df_eci['descuento'] = None
if 'valoracion' not in df_eci.columns:
    df_eci['valoracion'] = None
if 'fecha' not in df_eci.columns:
    df_eci['fecha'] = datetime.now().strftime("%Y-%m-%d %H:%M")

columnas_raw = ['nombre', 'precio_actual', 'precio_original', 'descuento', 'valoracion', 'plataforma', 'fecha']
df_eci_raw = df_eci[columnas_raw]

# 4. Limpiar los datos
try:
    df_eci_procesado = transform_prices(df_eci_raw.copy())
    
    ruta_procesado = project_root / "data" / "processed" / "prueba_eci_procesado.csv"
    df_eci_procesado.to_csv(ruta_procesado, index=False)
    
    print("✅ ¡ÉXITO! Jupyter ha leído el nuevo ETL.py")
    print(f"Filas procesadas con éxito: {len(df_eci_procesado)}")
    print("\n--- Muestra de cómo ha quedado el precio limpio ---")
    display(df_eci_procesado[['nombre', 'precio_actual_num', 'precio_original_num', 'descuento_pct', 'plataforma']].head())
    
except Exception as e:
    print(f"❌ ERROR en la limpieza ETL: {e}")

✅ ¡ÉXITO! Jupyter ha leído el nuevo ETL.py
Filas procesadas con éxito: 12

--- Muestra de cómo ha quedado el precio limpio ---


,nombre,precio_actual_num,precio_original_num,descuento_pct,plataforma
9,Impresora Multifunción Epson Expression Home X...,60.9,71.9,15.30,ElCorteIngles
5,"Impresora multifunción HP OfficeJet Pro 8134e,...",139.9,169.9,17.66,ElCorteIngles
4,"Monitor PC 68,5cm (27"") LG MyView Smart Monito...",179.0,219.0,18.26,ElCorteIngles
2,"Tablet Lenovo Idea Tab 8GB + 128GB, 11"", Wi-Fi...",199.0,NaN,NaN,ElCorteIngles
7,"Apple iPad 11"" (2025) A16, Wi-Fi",359.0,379.0,5.28,ElCorteIngles
